# Microbatch Repartitioning 

Notebook to run operations to support tickets: https://github.com/cal-itp/data-infra/issues/4865, https://github.com/cal-itp/data-infra/issues/4878, https://github.com/cal-itp/data-infra/issues/5063. Specifically, operations related to copying an existing (non-partitioned, non-microbatch) table and making a new partitioned version in the original location to transition a model to microbatch "in-place".

To run this, you need to auth: `gcloud auth application-default login --login-config=../../../iac/login.json` and select the Python kernel associated with this Poetry env. 

In [ ]:
# enter the models to be re-partitioned 
PROJECT = 'cal-itp-data-infra-staging'
SCHEMA = 'mart_gtfs'
MODELS = ['fct_daily_rt_feed_files']

# enter the partition info
PARTITION_COLUMN = 'date'

In [ ]:
# make a dictionary of {fully qualified name: fully qualified name for partitioned copy}
models_dict = {PROJECT + '.' + SCHEMA + '.' + table: PROJECT + '.' + SCHEMA + '.' + table + '_partitioned' for table in MODELS}

In [ ]:
from google.cloud import bigquery
client = bigquery.Client(PROJECT)

In [ ]:
# used this step for testing of dim_stop_times but deciding not to do this for subsequent models

# first, copy the current state of the model 
# for model_full_name in models_dict.keys():
#     orig_table_copy_name = model_full_name + '_orig'
#     job = client.copy_table(model_full_name, orig_table_copy_name)
#     job.result()
#     # check that row counts are equal 
#     orig_row_count = client.query(f"select count(*) from {model_full_name}").result()
#     copy_row_count = client.query(f"select count(*) from {orig_table_copy_name}").result()
#     assert next(orig_row_count) == next(copy_row_count), "copy and original should have same row count"
#     print(f"Copied {model_full_name} into {orig_table_copy_name}")

In [ ]:
# make a partitioned copy of the model
for model_full_name, partitioned_table in models_dict.items():

    job_config = bigquery.QueryJobConfig()
    job_config.destination = partitioned_table
    job_config.time_partitioning = bigquery.table.TimePartitioning(type_ = 'DAY', field = PARTITION_COLUMN)

    query = f"select * from {model_full_name}"

    client.query_and_wait(query, job_config=job_config)

    # check that row counts are equal 
    orig_row_count = client.query(f"select count(*) from {model_full_name}").result()
    partitioned_row_count = client.query(f"select count(*) from {partitioned_table}").result()

    assert next(orig_row_count) == next(partitioned_row_count), "copy and partitioned version should have same row count"

    print(f"Copied data into partitioned version model {partitioned_table}")


In [ ]:
# apply labels to partitioned tables
# from https://docs.cloud.google.com/bigquery/docs/adding-labels#python_1

for partitioned_table in models_dict.values():
    table = client.get_table(partitioned_table)
    labels = {'domain': 'mart', 'dataset': 'gtfs'}
    table.labels = labels
    table = client.update_table(table, ["labels"])
    print(f"Added {table.labels} to {partitioned_table}.")

In [ ]:
# now, delete the original 
for model_full_name in models_dict.keys():
    client.delete_table(model_full_name)
    print(f"Deleted {model_full_name}")

In [ ]:
# now copy the partitioned version into the original location
for model_full_name, partitioned_table in models_dict.items():
    job = client.copy_table(partitioned_table, model_full_name)
    job.result()

    print(f"Copied {partitioned_table} into {model_full_name}")

In [ ]:
# now, delete the partitioned copy 
for partitioned_table in models_dict.values():
    client.delete_table(partitioned_table)
    print(f"Deleted {partitioned_table}")